# v38 — 基于原始 33.89 baseline 的最小性能补丁

本 notebook 以 `baseline_LB33.89_failed-in-aimo.ipynb` 为母版：**保留原有 9 个 cell、原 metadata、模型路径、竞赛输入、4×L4 并行、128 条 task-time 训练增强、16 条推理增强、LoRA 参数、解码阈值、候选聚合和 submission schema。**

仅在原 `arc_solver.py` 的两个 logits 热点中做定点替换：

1. `turbo_dfs`：仍对同一组 12 个 ARC token 使用同样的 NLL、阈值、候选排序与 DFS；仅把 full-vocabulary logits 的归一化留在 GPU，并只把 12 个 token 的结果传回 CPU。
2. `calc_scores`：仍使用同一 teacher-forced NLL 作为重排序分数；仅在 GPU 上 gather 目标 token 并关闭未被消费的 KV cache。

没有缩短或延长原 `global_end_time = now + 12h − 10min`，也没有改动 `starter.py`、结果写入、队列策略或提交逻辑。该版本的目的仅是减少 CPU↔GPU logits 传输与多余 KV cache 写入，从而在**不增加时间预算**的前提下提高已完成任务覆盖率。

In [ ]:
# Keep the original global 10-minute submission/write buffer.
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
%%writefile dsl_all.py
"""ARC-AGI-2 DSL: grid primitives + objects + candidate solvers.

Design: `SOLVERS` is a list of functions f(train_pairs) -> Optional[callable g].
Each tries to find parameters such that g(input)==output for ALL train pairs.
The outer runner applies found g to test inputs.
"""
import numpy as np
from collections import Counter, deque
from itertools import product, permutations

Grid = np.ndarray

# ---------------------------------------------------------------- basics
def G(a): return np.asarray(a, dtype=int)
def rot(g, k): return np.rot90(g, k)
def fliph(g): return np.fliplr(g)
def flipv(g): return np.flipud(g)
def transp(g): return g.T
def colors(g): return set(np.unique(g))
def most_common(g, bg=None):
    c = Counter(g.ravel())
    if bg is not None and bg in c: del c[bg]
    return c.most_common(1)[0][0] if c else 0
def least_common(g):
    return Counter(g.ravel()).most_common()[-1][0]
def replace(g, a, b):
    out = g.copy(); out[g == a] = b; return out
def of_color(g, c): return np.argwhere(g == c)
def bbox_of(mask_or_cells):
    cells = np.argwhere(mask_or_cells) if mask_or_cells.dtype == bool else np.asarray(mask_or_cells)
    if len(cells) == 0: return None
    (y0, x0), (y1, x1) = cells.min(0), cells.max(0)
    return y0, x0, y1 + 1, x1 + 1
def crop(g, bb):
    y0, x0, y1, x1 = bb; return g[y0:y1, x0:x1]
def trim(g, bg=None):
    bg = most_common(g) if bg is None else bg
    bb = bbox_of(g != bg)
    return crop(g, bb) if bb else g
def tile_to(g, h, w):
    th, tw = g.shape
    out = np.tile(g, (int(np.ceil(h/th)), int(np.ceil(w/tw))))
    return out[:h, :w]
def upscale(g, k): return np.repeat(np.repeat(g, k, 0), k, 1)
def downscale(g, k):
    return g[::k, ::k]
def pad_to(g, h, w, c=0):
    out = np.full((h, w), c, g.dtype)
    h2, w2 = min(h, g.shape[0]), min(w, g.shape[1])
    out[:h2, :w2] = g[:h2, :w2]
    return out

# ---------------------------------------------------------------- objects
def components(g, bg=None, conn=4):
    """Connected components of non-bg cells -> list of (cells, color)."""
    bg = 0 if bg is None else bg
    H, W = g.shape
    seen = np.zeros_like(g, bool)
    out = []
    for y in range(H):
        for x in range(W):
            if seen[y, x] or g[y, x] == bg:
                continue
            c = g[y, x]
            q = deque([(y, x)]); seen[y, x] = True; cells = []
            while q:
                cy, cx = q.popleft(); cells.append((cy, cx))
                for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)) + ((1,1),(1,-1),(-1,1),(-1,-1) if conn==8 else ()):
                    ny, nx = cy+dy, cx+dx
                    if 0<=ny<H and 0<=nx<W and not seen[ny,nx] and g[ny,nx]==c:
                        seen[ny,nx]=True; q.append((ny,nx))
            out.append((cells, int(c)))
    return out

def components_multicolor(g, bg=0, conn=4):
    """Components ignoring color."""
    H, W = g.shape
    seen = np.zeros_like(g, bool)
    out = []
    nb = ((1,0),(-1,0),(0,1),(0,-1)) + (((1,1),(1,-1),(-1,1),(-1,-1)) if conn==8 else ())
    for y in range(H):
        for x in range(W):
            if seen[y, x] or g[y, x] == bg: continue
            q = deque([(y,x)]); seen[y,x]=True; cells=[]
            while q:
                cy,cx=q.popleft(); cells.append((cy,cx))
                for dy,dx in nb:
                    ny,nx=cy+dy,cx+dx
                    if 0<=ny<H and 0<=nx<W and not seen[ny,nx] and g[ny,nx]!=bg:
                        seen[ny,nx]=True; q.append((ny,nx))
            out.append(cells)
    return out

def normalize(obj_cells):
    ys=[c[0] for c in obj_cells]; xs=[c[1] for c in obj_cells]
    y0,x0=min(ys),min(xs)
    return frozenset((y-y0,x-x0) for y,x in obj_cells)

def obj_grid(cells, color):
    ys=[c[0] for c in cells]; xs=[c[1] for c in cells]
    g=np.zeros((max(ys)+1-min(ys), max(xs)+1-min(xs)), int)
    for y,x in cells: g[y-min(ys),x-min(xs)]=color
    return g

def gravity(g, bg=0, dir='down'):
    out = np.full_like(g, bg)
    H, W = g.shape
    if dir in ('down','up'):
        for x in range(W):
            col = [g[y,x] for y in range(H) if g[y,x]!=bg]
            ys = range(H-len(col),H) if dir=='down' else range(len(col))
            for y,c in zip(ys,col): out[y,x]=c
    else:
        for y in range(H):
            row=[g[y,x] for x in range(W) if g[y,x]!=bg]
            xs = range(W-len(row),W) if dir=='right' else range(len(row))
            for x,c in zip(xs,row): out[y,x]=c
    return out

def fill_enclosed(g, bg=0, color=1):
    """Flood-fill background from borders; interior bg -> color."""
    out=g.copy(); H,W=g.shape
    seen=np.zeros_like(g,bool); q=deque()
    for y in range(H):
        for x in (0,W-1):
            if g[y,x]==bg and not seen[y,x]: seen[y,x]=True; q.append((y,x))
    for x in range(W):
        for y in (0,H-1):
            if g[y,x]==bg and not seen[y,x]: seen[y,x]=True; q.append((y,x))
    while q:
        y,x=q.popleft()
        for dy,dx in DIRS:
            ny,nx=y+dy,x+dx
            if 0<=ny<H and 0<=nx<W and not seen[ny,nx] and g[ny,nx]==bg:
                seen[ny,nx]=True; q.append((ny,nx))
    out[(g==bg)&(~seen)]=color
    return out
DIRS=((-1,0),(1,0),(0,-1),(0,1))

"""Solver battery for ARC-AGI-2. Each solver takes list of (in,out) pairs and
returns a callable g(grid)->grid valid on all pairs, or None."""
import numpy as np
from collections import Counter, deque
from itertools import product, permutations, combinations


def ok_all(fn, pairs):
    try:
        for a, b in pairs:
            r = fn(G(a))
            if r is None or not np.array_equal(np.asarray(r), np.asarray(b)):
                return False
        return True
    except Exception:
        return False

def wrap(fn):
    """Make a solver-fn into (pairs)->callable or None."""
    def s(pairs):
        f = fn(pairs)
        return f
    return s

# ---------------- individual hypothesis families ----------------

def s_geom(pairs):
    for t in [lambda g: rot(g,1), lambda g: rot(g,2), lambda g: rot(g,3),
              fliph, flipv, transp, lambda g: np.fliplr(transp(g)),
              lambda g: transp(fliph(g)), lambda g: rot(transp(g),1)]:
        if ok_all(t, pairs):
            return t
    return None

def s_identity_plus(pairs):
    cands = []
    # color permutation learned from first pair
    def learn_map(a, b):
        if a.shape != b.shape: return None
        m = {}
        for c in np.unique(a):
            tgt = b[a == c]
            if len(set(np.unique(tgt))) == 1:
                m[int(c)] = int(tgt.flat[0])
            else:
                return None
        return m
    m = learn_map(G(pairs[0][0]), G(pairs[0][1]))
    if m is not None:
        def f(g, m=m):
            out = g.copy()
            for k, v in m.items(): out[g == k] = v
            return out
        if ok_all(f, pairs): return f
    # replace each color with most-common-of-output
    return None

def s_recolor_const(pairs):
    """All cells of color a -> b (single substitution)."""
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    if a0.shape != b0.shape: return None
    diff = a0 != b0
    if not diff.any(): return None
    cs = np.unique(a0[diff]); ts = np.unique(b0[diff])
    for c in cs:
        for t in set(np.unique(b0)) | set(np.unique(a0)):
            f = lambda g, c=c, t=t: replace(g, c, t)
            if ok_all(f, pairs): return f
    return None

def s_trim(pairs):
    for bgname, bg in [("auto", None)] + [(f"c{c}", c) for c in range(10)]:
        def f(g, bg=bg):
            bb = bg if bg is not None else most_common(g)
            return trim(g, bb)
        if ok_all(f, pairs): return f
    return None

def s_crop_color_bbox(pairs):
    for c in range(10):
        def f(g, c=c):
            m = g == c
            if not m.any(): return None
            return crop(g, bbox_of(m))
        if ok_all(f, pairs): return f
    return None

def s_upscale(pairs):
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    if a0.size and b0.shape[0] % a0.shape[0] == 0 and b0.shape[1] % a0.shape[1] == 0:
        k = b0.shape[0] // a0.shape[0]
        if k > 1 and b0.shape[1] // a0.shape[1] == k:
            for mode in range(3):
                if mode == 0:
                    f = lambda g, k=k: upscale(g, k)
                elif mode == 1:  # pixel -> block of itself, but in pattern of input
                    def f(g, k=k):
                        h, w = g.shape
                        out = np.zeros((h*k, w*k), int)
                        for y in range(h):
                            for x in range(w):
                                out[y*k:(y+1)*k, x*k:(x+1)*k] = g[y, x]
                        return out
                else:  # each pixel becomes scaled input masked by color
                    def f(g, k=k):
                        h, w = g.shape
                        base = upscale(g, k)
                        return base
                if ok_all(f, pairs): return f
    return None

def mirror_tile(g, oh, ow, flip_v=True, flip_h=True):
    th, tw = g.shape
    out = np.zeros((oh, ow), g.dtype)
    for by in range(int(np.ceil(oh/th))):
        for bx in range(int(np.ceil(ow/tw))):
            blk = g
            if flip_v and by % 2: blk = np.flipud(blk)
            if flip_h and bx % 2: blk = np.fliplr(blk)
            y0, x0 = by*th, bx*tw
            blk = blk[:min(th, oh-y0), :min(tw, ow-x0)]
            out[y0:y0+blk.shape[0], x0:x0+blk.shape[1]] = blk
    return out

def s_tile(pairs):
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    if b0.shape[0] % a0.shape[0] == 0 and b0.shape[1] % a0.shape[1] == 0:
        f = lambda g: tile_to(g, *G(pairs[0][1]).shape)
        if ok_all(f, pairs): return f
    # tile to double/triple
    for k in (2, 3, 4):
        f = lambda g, k=k: tile_to(g, g.shape[0]*k, g.shape[1]*k)
        if ok_all(f, pairs): return f
    # mirror tiling (alternating flips)
    for fv, fh in [(True, False), (False, True), (True, True)]:
        def f(g, fv=fv, fh=fh):
            a = G(pairs[0][0]); b = G(pairs[0][1])
            ky = b.shape[0] // a.shape[0]; kx = b.shape[1] // a.shape[1]
            return mirror_tile(g, g.shape[0]*ky, g.shape[1]*kx, fv, fh)
        if ok_all(f, pairs): return f
    return None

def s_kron_self(pairs):
    """Each nonzero input pixel -> copy of input (kron(mask, input) variants)."""
    def f1(g):
        m = (g != 0).astype(int)
        return np.kron(m, g)
    def f2(g):
        m = (g != 0).astype(int)
        return np.kron(g, np.ones_like(g)) * 0  # placeholder
    def f3(g):
        h, w = g.shape
        out = np.zeros((h*h, w*w), g.dtype)
        for y in range(h):
            for x in range(w):
                if g[y, x] != 0:
                    out[y*h:(y+1)*h, x*w:(x+1)*w] = np.where(g != 0, g[y, x], 0)
        return out
    for f in [f1, f3]:
        if ok_all(f, pairs): return f
    return None

def s_symmetry(pairs):
    """Complete symmetry along h/v/rot axes."""
    def sym_h(g):
        h, w = g.shape
        out = g.copy()
        for y in range(h):
            for x in range(w):
                if out[y, x] == 0:
                    out[y, x] = g[y, w-1-x]
        return out
    def sym_v(g):
        h, w = g.shape
        out = g.copy()
        for y in range(h):
            for x in range(w):
                if out[y, x] == 0:
                    out[y, x] = g[h-1-y, x]
        return out
    def sym_d(g):
        h, w = g.shape
        out = g.copy()
        for y in range(h):
            for x in range(w):
                if out[y, x] == 0:
                    out[y, x] = g[h-1-y, w-1-x]
        return out
    for f in [sym_h, sym_v, sym_d,
              lambda g: np.maximum(g, np.fliplr(g)),
              lambda g: np.maximum(g, np.flipud(g)),
              lambda g: np.maximum(g, np.rot90(g, 2)),
              lambda g: np.maximum.reduce([g, np.fliplr(g), np.flipud(g), np.rot90(g,2)])]:
        if ok_all(f, pairs): return f
    return None

def s_gravity(pairs):
    for d in ('down', 'up', 'left', 'right'):
        f = lambda g, d=d: gravity(g, 0, d)
        if ok_all(f, pairs): return f
    return None

def s_fill_enclosed(pairs):
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    newc = [c for c in np.unique(b0) if c not in np.unique(a0)]
    for c in list(newc) + list(range(1, 10)):
        f = lambda g, c=c: fill_enclosed(g, 0, c)
        if ok_all(f, pairs): return f
    return None

def s_denoise(pairs):
    """Remove isolated cells / keep only large components."""
    def f(g):
        out = np.zeros_like(g)
        for cells, c in components(g, bg=0, conn=4):
            if len(cells) > 1:
                for y, x in cells: out[y, x] = c
        return out
    if ok_all(f, pairs): return f
    def f2(g):
        out = np.zeros_like(g)
        comps = components(g, bg=0, conn=8)
        if not comps: return g
        big = max(len(c) for c, _ in comps)
        for cells, c in comps:
            if len(cells) == big:
                for y, x in cells: out[y, x] = c
        return out
    if ok_all(f2, pairs): return f2
    return None

def s_extract_object(pairs):
    """Output = bbox-crop of object chosen by rank (largest/smallest/etc)."""
    for conn in (4, 8):
        for crit in ("largest", "smallest", "rarecolor"):
            def f(g, conn=conn, crit=crit):
                comps = components(g, bg=0, conn=conn)
                if not comps: return None
                if crit == "largest": cells, c = max(comps, key=lambda t: len(t[0]))
                elif crit == "smallest": cells, c = min(comps, key=lambda t: len(t[0]))
                else:
                    cnt = Counter(g[g != 0].ravel())
                    rc = cnt.most_common()[-1][0]
                    sel = [t for t in comps if t[1] == rc]
                    if not sel: return None
                    cells, c = max(sel, key=lambda t: len(t[0]))
                return obj_grid(cells, c)
            if ok_all(f, pairs): return f
    return None

def s_recolor_by_size(pairs):
    """Recolor objects based on size rank."""
    def f(g):
        out = g.copy()
        comps = components(g, bg=0, conn=4)
        sizes = sorted(set(len(c) for c, _ in comps))
        for cells, c in comps:
            pass
        return out
    return None

def s_subgrid_split(pairs):
    """Split on a separator color into subgrids; combine or select."""
    def split(g, sep):
        m = g != sep
        rows = np.where(m.any(1))[0]
        cols = np.where(m.any(0))[0]
        # find full separator rows/cols
        rowsep = [y for y in range(g.shape[0]) if (g[y] == sep).all()]
        colsep = [x for x in range(g.shape[1]) if (g[:, x] == sep).all()]
        return rowsep, colsep
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    for sep in range(10):
        rs, cs = split(a0, sep)
        if not rs and not cs: continue
        # extract blocks
        def blocks(g, rs, cs, sep):
            rb = [-1] + rs + [g.shape[0]]; cb = [-1] + cs + [g.shape[1]]
            gs = []
            for i in range(len(rb)-1):
                row = []
                for j in range(len(cb)-1):
                    blk = g[rb[i]+1:rb[i+1], cb[j]+1:cb[j+1]]
                    row.append(blk)
                gs.append(row)
            return gs
        blks = blocks(a0, rs, cs, sep)
        flat = [b for row in blks for b in row if b.size]
        # try: combine via elementwise ops (needs equal shapes)
        if len(flat) >= 2 and all(b.shape == flat[0].shape for b in flat):
            for op in [np.maximum, np.minimum,
                       lambda x, y: np.where(x > 0, x, y),
                       lambda x, y: np.where((x > 0) & (y > 0), x, 0),
                       lambda x, y: np.where(x == y, x, 0),
                       lambda x, y: np.where(x != y, np.maximum(x, y), 0)]:
                def f(g, op=op, rs=None, cs=None, sep=sep):
                    rr, cc = split(g, sep)
                    bl = [b for row in blocks(g, rr, cc, sep) for b in row if b.size]
                    if not bl: return None
                    out = bl[0]
                    for b in bl[1:]:
                        if b.shape != out.shape: return None
                        out = op(out, b)
                    return out
                if ok_all(f, pairs): return f
    return None

def s_draw_lines(pairs):
    """Connect pairs of same-colored cells with lines."""
    def f(g):
        out = g.copy()
        for c in np.unique(g):
            if c == 0: continue
            ys, xs = np.where(g == c)
            if len(ys) == 2:
                y0, y1 = sorted(ys); x0, x1 = sorted(xs)
                if y0 == y1: out[y0, x0:x1+1] = c
                elif x0 == x1: out[y0:y1+1, x0] = c
                else:
                    for y in range(y0, y1+1): out[y, x0] = c
                    for x in range(x0, x1+1): out[y1, x] = c
        return out
    if ok_all(f, pairs): return f
    return None

def s_frame_border(pairs):
    def f(g):
        out = g.copy()
        c = most_common(g[g != 0]) if (g != 0).any() else 1
        out[0, :] = c; out[-1, :] = c; out[:, 0] = c; out[:, -1] = c
        return out
    if ok_all(f, pairs): return f
    return None

def s_complete_pattern(pairs):
    """Fill 0-cells using periodic repetition inferred from the row/col."""
    def f(g):
        h, w = g.shape
        out = g.copy()
        # find smallest period in each direction
        for axis in (0, 1):
            n = h if axis == 0 else w
            for p in range(1, n):
                ok = True
                for i in range(n - p):
                    a = g[i] if axis == 0 else g[:, i]
                    b = g[i+p] if axis == 0 else g[:, i+p]
                    m = (a != 0) & (b != 0)
                    if m.any() and not (a[m] == b[m]).all():
                        ok = False; break
                if ok:
                    for i in range(n):
                        src = i % p
                        if axis == 0:
                            row = g[src]
                            mask = out[i] == 0
                            out[i][mask] = row[mask]
                        else:
                            col = g[:, src]
                            mask = out[:, i] == 0
                            out[:, i][mask] = col[mask]
                    break
        return out
    if ok_all(f, pairs): return f
    return None

def s_extract_marked(pairs):
    """Extract subgrid marked by a distinctive colored marker/corner."""
    return None

def s_move_to_marker(pairs):
    return None

def s_fold(pairs):
    """Fold grid along mid axis combining halves."""
    def make(op):
        def f(g):
            h, w = g.shape
            for axis in (0, 1):
                n = h if axis == 0 else w
                if n % 2 == 1:
                    a = g[:n//2] if axis == 0 else g[:, :n//2]
                    b = (g[n//2+1:][::-1] if axis == 0 else g[:, n//2+1:][:, ::-1])
                    try:
                        return op(a, b)
                    except Exception:
                        return None
            return None
        return f
    for op in [np.maximum, np.minimum, lambda a,b: np.where(a!=0,a,b),
               lambda a,b: np.where((a!=0)&(b!=0), np.maximum(a,b), 0)]:
        f = make(op)
        if ok_all(f, pairs): return f
    return None

def s_count_repeat(pairs):
    """Output = input or pattern repeated/cropped per object count."""
    return None

def s_bbox_overlay(pairs):
    """Draw bounding box around each object / fill it."""
    def f(g):
        out = g.copy()
        for cells, c in components(g, bg=0, conn=8):
            y0,x0,y1,x1 = bbox_of(cells)
            out[y0:y1, x0:x1] = np.where(out[y0:y1,x0:x1]==0, c, out[y0:y1,x0:x1])
        return out
    if ok_all(f, pairs): return f
    return None

def s_swap_to_common(pairs):
    """Set bg to most common color overall."""
    def f(g):
        out = g.copy()
        c = most_common(g)
        out[g == 0] = c
        return out
    if ok_all(f, pairs): return f
    return None

def s_cellwise_binary(pairs):
    """output = binary/derived function of input mask (e.g., invert, edges)."""
    def f1(g):
        return np.where(g == 0, np.max(g) if np.max(g)>0 else 1, 0)
    for f in [f1,
              lambda g: np.where(g==0, most_common(g), 0),
              lambda g: np.where(g!=0, most_common(g), 0)]:
        if ok_all(f, pairs): return f
    return None

def s_keep_color(pairs):
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    keep = set(np.unique(b0)) & set(np.unique(a0))
    for c in keep:
        f = lambda g, c=c: np.where(g == c, c, 0)
        if ok_all(f, pairs): return f
    return None

def s_extend_lines(pairs):
    """Extend existing line segments across the grid."""
    def f(g):
        out = g.copy(); H, W = g.shape
        for c in np.unique(g):
            if c == 0: continue
            ys, xs = np.where(g == c)
            if len(ys) < 2: continue
            if len(set(ys)) == 1: out[ys[0], :] = c
            elif len(set(xs)) == 1: out[:, xs[0]] = c
        return out
    if ok_all(f, pairs): return f
    return None

def s_project(pairs):
    """Project objects onto axis (rows/cols histogram as pattern)."""
    return None

def s_copy_per_object(pairs):
    """Each object gets its own subgrid region (object separation)."""
    return None

def s_fill_holes(pairs):
    """Fill holes inside each object with a learned/selected color."""
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    newc = [c for c in np.unique(b0) if c not in np.unique(a0)] or list(range(10))
    for c in list(newc) + list(range(10)):
        def f(g, c=c):
            out = g.copy()
            bg = most_common(g)
            for cells, col in components(g, bg=bg, conn=4):
                ys = [x[0] for x in cells]; xs = [x[1] for x in cells]
                y0, y1, x0, x1 = min(ys), max(ys)+1, min(xs), max(xs)+1
                sub = g[y0:y1, x0:x1]
                mask = np.ones_like(sub, bool)
                for (cy, cx) in cells: mask[cy-y0, cx-x0] = False
                # flood fill from border of sub
                seen = np.zeros_like(sub, bool); q = deque()
                for yy in range(sub.shape[0]):
                    for xx in (0, sub.shape[1]-1):
                        if mask[yy, xx] and not seen[yy, xx]: seen[yy, xx]=True; q.append((yy,xx))
                for xx in range(sub.shape[1]):
                    for yy in (0, sub.shape[0]-1):
                        if mask[yy, xx] and not seen[yy, xx]: seen[yy, xx]=True; q.append((yy,xx))
                while q:
                    cy, cx = q.popleft()
                    for dy, dx in DIRS:
                        ny, nx = cy+dy, cx+dx
                        if 0<=ny<sub.shape[0] and 0<=nx<sub.shape[1] and mask[ny,nx] and not seen[ny,nx]:
                            seen[ny,nx]=True; q.append((ny,nx))
                holes = mask & ~seen
                for hy, hx in np.argwhere(holes):
                    out[y0+hy, x0+hx] = c
            return out
        if ok_all(f, pairs): return f
    return None

def s_odd_one_out(pairs):
    """Find the object whose shape differs; output it or use it as mask."""
    for conn in (4, 8):
        for mode in ('extract', 'recolor'):
            def f(g, conn=conn, mode=mode):
                comps = components(g, bg=0, conn=conn)
                if len(comps) < 2: return None
                # group by normalized shape+color
                from collections import defaultdict
                groups = defaultdict(list)
                for cells, c in comps:
                    groups[(normalize(cells), c)].append((cells, c))
                odd = [v[0] for v in groups.values() if len(v) == 1]
                if len(odd) != 1: return None
                cells, c = odd[0]
                if mode == 'extract':
                    return obj_grid(cells, c)
                out = g.copy()
                nc = least_common(g)
                for y, x in cells: out[y, x] = nc if nc else 1
                return out
            if ok_all(f, pairs): return f
    return None

def s_color_count_square(pairs):
    """Output = NxN or 1xN grid of color c where N = count of something."""
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    oh, ow = b0.shape
    if oh == ow or ow == 1 or oh == 1:
        n = max(oh, ow)
        for what in ('nobj', 'ncolors'):
            for c in range(10):
                def f(g, n0=n, c=c, what=what):
                    if what == 'nobj':
                        nn = len(components_multicolor(g, bg=0))
                    else:
                        nn = len(colors(g) - {0})
                    if oh == ow:
                        return np.full((nn, nn), c, int)
                    return np.full((1, nn) if oh == 1 else (nn, 1), c, int)
                if ok_all(f, pairs): return f
    return None

def s_marker_line(pairs):
    """Draw line between the two marker cells of a distinctive color."""
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    cand = [c for c in np.unique(a0) if c != 0]
    for c in cand:
        for lc in range(1, 10):
            def f(g, c=c, lc=lc):
                pts = np.argwhere(g == c)
                if len(pts) != 2: return None
                out = g.copy()
                (y0, x0), (y1, x1) = pts
                if y0 == y1:
                    out[y0, min(x0,x1):max(x0,x1)+1] = lc
                elif x0 == x1:
                    out[min(y0,y1):max(y0,y1)+1, x0] = lc
                elif abs(y1-y0) == abs(x1-x0):
                    n = abs(y1-y0)
                    for i in range(n+1):
                        out[y0 + i*(1 if y1>y0 else -1), x0 + i*(1 if x1>x0 else -1)] = lc
                else:
                    return None
                return out
            if ok_all(f, pairs): return f
    return None

def s_recolor_objects_size(pairs):
    """Recolor objects by size: largest->c1, smallest->c2 etc."""
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    outc = [c for c in np.unique(b0) if c != 0]
    for c_big in outc:
        for c_small in outc:
            def f(g, c_big=c_big, c_small=c_small):
                comps = components(g, bg=0, conn=8)
                if len(comps) < 2: return None
                out = g.copy()
                big = max(comps, key=lambda t: len(t[0]))
                small = min(comps, key=lambda t: len(t[0]))
                for cells, c in comps:
                    nc = c_big if (cells, c) == big else (c_small if (cells, c) == small else c)
                    for y, x in cells: out[y, x] = nc
                return out
            if ok_all(f, pairs): return f
    return None

def s_repeat_dir(pairs):
    """Repeat a detected motif to fill the grid along an axis."""
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    if a0.shape != b0.shape: return None
    def f(g):
        out = g.copy(); H, W = g.shape
        for c in np.unique(g):
            if c == 0: continue
            ys, xs = np.where(g == c)
            if len(ys) == 0: continue
            # horizontal repeat
            if len(set(ys)) == 1 and (g == c).sum() < W:
                y = ys[0]
                xs_sorted = sorted(set(xs))
                if len(xs_sorted) >= 2:
                    period = xs_sorted[1] - xs_sorted[0]
                    x = xs_sorted[0]
                    while x < W:
                        out[y, x] = c; x += period
            if len(set(xs)) == 1 and (g == c).sum() < H:
                x = xs[0]
                ys_sorted = sorted(set(ys))
                if len(ys_sorted) >= 2:
                    period = ys_sorted[1] - ys_sorted[0]
                    y = ys_sorted[0]
                    while y < H:
                        out[y, x] = c; y += period
        return out
    if ok_all(f, pairs): return f
    return None

def s_enclose_box(pairs):
    """Draw minimal rectangle border around objects of a certain color."""
    a0, b0 = G(pairs[0][0]), G(pairs[0][1])
    for c in range(10):
        for lc in range(1, 10):
            def f(g, c=c, lc=lc):
                out = g.copy()
                for cells, col in components(g, bg=0, conn=8):
                    if col != c: continue
                    y0, x0, y1, x1 = bbox_of(cells)
                    out[y0, x0:x1] = lc; out[y1-1, x0:x1] = lc
                    out[y0:y1, x0] = lc; out[y0:y1, x1-1] = lc
                return out
            if ok_all(f, pairs): return f
    return None

def s_majority_vote_quarters(pairs):
    """Split into quadrants/subgrids and take majority/union."""
    return None

def s_extract_inner(pairs):
    """Extract interior of the largest frame-like object."""
    for conn in (4, 8):
        def f(g, conn=conn):
            comps = components(g, bg=0, conn=conn)
            if not comps: return None
            cells, c = max(comps, key=lambda t: len(t[0]))
            y0, x0, y1, x1 = bbox_of(cells)
            inner = g[y0+1:y1-1, x0+1:x1-1]
            return inner if inner.size else None
        if ok_all(f, pairs): return f
    return None

SOLVERS = [s_geom, s_identity_plus, s_recolor_const, s_trim, s_crop_color_bbox,
           s_upscale, s_tile, s_kron_self, s_symmetry, s_gravity, s_fill_enclosed,
           s_denoise, s_extract_object, s_subgrid_split, s_draw_lines,
           s_frame_border, s_complete_pattern, s_fold, s_bbox_overlay,
           s_swap_to_common, s_cellwise_binary, s_keep_color, s_extend_lines,
           s_fill_holes, s_odd_one_out, s_color_count_square, s_marker_line,
           s_recolor_objects_size, s_repeat_dir, s_enclose_box, s_extract_inner]


In [ ]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

decoder.load_decoded_results("/kaggle/inference_outputs")

results = decoder.run_selection_algo()
# --- DSL injection: verified symbolic predictions become attempt_1 ---
import sys as _sys
_sys.path.insert(0, '/kaggle/working')
try:
    from dsl_all import SOLVERS as _SOLVERS
    import numpy as _np
    _inj = 0
    for _k, _task in data.queries.items():
        _pairs = [(_p['input'], _p['output']) for _p in _task['train']]
        for _i, _tt in enumerate(_task['test']):
            _bk = f'{_k}_{_i}'
            _preds = []
            for _s in _SOLVERS:
                try:
                    _f = _s(_pairs)
                    if _f:
                        _r = _np.asarray(_f(_np.asarray(_tt['input'], dtype=int)))
                        if _r is not None and _r.ndim == 2 and 0 < _r.shape[0] <= 30 and 0 < _r.shape[1] <= 30:
                            _preds.append(_r)
                except Exception:
                    pass
            _uniq = []
            for _p in _preds:
                if not any(_np.array_equal(_p, _q) for _q in _uniq):
                    _uniq.append(_p)
            if _uniq:
                if _bk in results:
                    results[_bk] = _uniq[:1] + list(results[_bk])
                else:
                    results[_bk] = _uniq
                _inj += 1
    print('*** DSL injected for', _inj, 'test inputs')
except Exception as _e:
    print('DSL injection skipped:', _e)
submission = data.get_submission(results)

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))